                # 03 Model Training Lab

                Purpose: pull latest code, run baseline models, train tabular
                Phase 1 models, write model cards and summary predictions, and
                generate calibration diagnostics. This is not a backtest.
                


## 1. Mount Drive And Project Root


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import sys
from pathlib import Path

# Drive is the persistent research/artifact root.
PROJECT_ROOT = "/content/drive/MyDrive/btcusdt_quant_research"

# Code is checked out on the Colab VM, not inside Drive. This avoids stale Drive git state.
REPO_DIR = "/content/btcusdt_quant_research_repo"

Path(PROJECT_ROOT).mkdir(parents=True, exist_ok=True)
for rel in [
    "data/raw", "data/processed", "data/feature_store", "data/labels", "data/events",
    "models/checkpoints", "models/trained_models", "models/calibration", "models/feature_importance",
    "reports/experiments", "logs", "cache",
]:
    Path(PROJECT_ROOT, rel).mkdir(parents=True, exist_ok=True)

# Keep the prompt-required Drive src path available, but imports should prefer the fresh VM checkout.
if f"{PROJECT_ROOT}/src" not in sys.path:
    sys.path.append(f"{PROJECT_ROOT}/src")
print("Drive artifact root:", PROJECT_ROOT)
print("Colab code checkout:", REPO_DIR)


## 2. Pull Latest Code


In [ ]:
import subprocess
from pathlib import Path

GITHUB_REPO_URL = "https://github.com/umutergul74/btcusdt_quant_research.git"
GITHUB_BRANCH = "main"
repo_path = Path(REPO_DIR)


def run_command(command, *, check=True):
    proc = subprocess.run(command, text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout.strip())
    if proc.stderr:
        print(proc.stderr.strip())
    if check and proc.returncode != 0:
        raise RuntimeError(
            "Command failed with exit code "
            f"{proc.returncode}: {' '.join(command)}"
        )
    return proc


if (repo_path / ".git").exists():
    run_command(["git", "-C", REPO_DIR, "remote", "set-url", "origin", GITHUB_REPO_URL])
    run_command(["git", "-C", REPO_DIR, "fetch", "origin", GITHUB_BRANCH])
    run_command(["git", "-C", REPO_DIR, "checkout", "-B", GITHUB_BRANCH, f"origin/{GITHUB_BRANCH}"])
else:
    if repo_path.exists() and any(repo_path.iterdir()):
        raise RuntimeError(
            f"{REPO_DIR} exists but is not an empty git checkout. "
            "Restart the Colab runtime or remove that temporary folder before continuing."
        )
    run_command(["git", "clone", "--branch", GITHUB_BRANCH, "--single-branch", GITHUB_REPO_URL, REPO_DIR])

repo_commit = run_command(["git", "-C", REPO_DIR, "rev-parse", "HEAD"]).stdout.strip()
repo_branch = run_command(["git", "-C", REPO_DIR, "branch", "--show-current"]).stdout.strip()
assert repo_branch == GITHUB_BRANCH, f"Expected {GITHUB_BRANCH}, found {repo_branch}"

if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")
print("Repository branch:", repo_branch)
print("Repository commit:", repo_commit)
print("After a changed checkout, use Runtime -> Restart session before trusting already-imported modules.")


## 3. Install Dependencies And Stage Helper


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt"], check=True)

from pprint import pprint
from btc_quant.pipeline.stages import run_stage
from btc_quant.pipeline.stage_registry import list_stage_specs

RUN_ID = "debug_colab"


def run_stage_checked(stage_name, config_name=None):
    print(f"\n=== {stage_name} ===")
    result = run_stage(stage_name, config_name=config_name, project_root=PROJECT_ROOT, run_id=RUN_ID)
    pprint(result)
    if result.get("status") == "FAIL":
        raise RuntimeError(f"Stage failed: {stage_name}")
    return result


## 4. Review Model Configuration


In [ ]:
from pathlib import Path
print((Path(REPO_DIR) / "configs" / "models.yaml").read_text())


## 5. Run Baseline Models


In [ ]:
baseline_result = run_stage_checked("train_baselines", config_name="models")


## 6. Train Tabular ML Models


In [ ]:
tabular_result = run_stage_checked("train_tabular", config_name="models")


## 7. Run Calibration Diagnostics


In [ ]:
calibration_result = run_stage_checked("calibrate_models", config_name="models")


## 8. Inspect Leaderboards And Calibration


In [ ]:
from pathlib import Path
import pandas as pd

experiment_dir = Path(PROJECT_ROOT) / "reports" / "experiments" / RUN_ID
print("\nBaseline comparison")
print(pd.read_csv(experiment_dir / "baseline_comparison.csv"))
print("\nModel leaderboard")
print(pd.read_csv(experiment_dir / "model_leaderboard.csv"))
print("\nCalibration metrics")
print(pd.read_csv(experiment_dir / "calibration_metrics.csv"))


## 9. Confirm Phase Boundary


In [ ]:
print("Training complete for Phase 1 diagnostics.")
print("No equity curve, drawdown, position sizing, stop-loss, take-profit, or trade journal was produced.")


In [ ]:
# Auto-close the Colab runtime after every previous cell has completed.
# Set this to False before running if you want to keep the session alive for inspection.
AUTO_CLOSE_COLAB_RUNTIME = True

if AUTO_CLOSE_COLAB_RUNTIME:
    try:
        from google.colab import runtime
        print("All notebook cells completed. Releasing the Colab runtime now.")
        runtime.unassign()
    except Exception as exc:
        print(f"Auto-close skipped outside Colab or because runtime release failed: {exc}")
